In [ ]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import numpy as np


from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from joblib import dump, load


In [ ]:
DATA_CSV = Path('../dataset_generators/datasets/final_1_lag_ffa_dataset.csv')
df_sample = pd.read_csv(DATA_CSV, nrows=5)
print('Dataset shape (sample inspected):', pd.read_csv(DATA_CSV).shape)
print('Columns:', list(pd.read_csv(DATA_CSV, nrows=0).columns))
display(df_sample.head())

def choose_target_column(columns):
    if 'fantasy_points_r_avg_1' in columns:
        return 'fantasy_points_r_avg_1'
    if 'fantasy_points' in columns:
        return 'fantasy_points'

    for c in columns:
        if 'fantasy_points' in c:
            return c
    return None

cols = list(pd.read_csv(DATA_CSV, nrows=0).columns)
TARGET_COLUMN = choose_target_column(cols)
print('Selected target column:', TARGET_COLUMN)

In [24]:
def build_pipeline(X: pd.DataFrame, alpha: float = 1.0) -> Pipeline:
    """Create a scikit-learn Pipeline that imputes and scales numeric features and fits Ridge."""
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    num_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="mean")),
        ("scale", StandardScaler()),
    ])
    pre = ColumnTransformer([("num", num_pipe, num_cols)])
    pipe = Pipeline([("pre", pre), ("est", Ridge(alpha=alpha))])
    return pipe

In [ ]:
def train_ridge(X: pd.DataFrame, y: pd.Series, alpha: float = 1.0) -> Pipeline:
    """Fit a Ridge regression pipeline on X and y and return the fitted pipeline."""
    pipe = build_pipeline(X, alpha=alpha)
    pipe.fit(X, y)
    return pipe

def fit_and_save(csv_path: Path | str, model_path: Path | str, target: str = 'fantasy_points_r_avg_1', alpha: float = 1.0) -> Pipeline:
    """Read CSV at csv_path, preprocess, train a Ridge model and save it to model_path.

    Returns the fitted pipeline."""
    csv_path = Path(csv_path)
    model_path = Path(model_path)
    df = pd.read_csv(csv_path)

    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in {csv_path}")

    df_proc = preprocess_df(df)

    if target not in df_proc.columns:
        df_proc[target] = df[target]

    X = df_proc.drop(columns=[target])
    y = df_proc[target]

    model = train_ridge(X, y, alpha=alpha)
    dump(model, model_path)
    return model

In [40]:
def load_model(model_path: Path | str):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}")
    return load(model_path)

def predict_dataframe(model, df: pd.DataFrame) -> pd.Series:
    df_proc = preprocess_df(df)
    return pd.Series(model.predict(df_proc), index=df.index)

In [42]:

from sklearn.model_selection import train_test_split
from sklearn import metrics

DATA_CSV = Path('../dataset_generators/datasets/final_1_lag_ffa_dataset.csv')
MODEL_OUT = Path('model.joblib')


df = pd.read_csv(DATA_CSV)
print('Loaded dataset with shape:', df.shape)

if 'fantasy_points_r_avg_1' in df.columns:
    target = 'fantasy_points_r_avg_1'
elif 'fantasy_points' in df.columns:
    target = 'fantasy_points'
else:
    cand = [c for c in df.columns if 'fantasy_points' in c]
    target = cand[0] if cand else None

if target is None:
    raise RuntimeError('No fantasy-points-like column found in dataset; please set target manually.')

print('Using target:', target)
df_proc = preprocess_df(df)

if target not in df_proc.columns:
    df_proc[target] = df[target]

X = df_proc.drop(columns=[target])
y = df_proc[target]

feature_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print('Number of numeric feature columns:', len(feature_cols))
print('Sample feature columns:', feature_cols[:20])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = train_ridge(X_train, y_train, alpha=1.0)

preds = model.predict(X_test)
r2 = metrics.r2_score(y_test, preds)
mse = metrics.mean_squared_error(y_test, preds)
rmse = mse ** 0.5
print(f'R^2: {r2:.4f}, RMSE: {rmse:.4f}')

dump(model, MODEL_OUT)
print('Saved trained model to', MODEL_OUT)


Loaded dataset with shape: (60969, 78)
Using target: fantasy_points
Number of numeric feature columns: 68
Sample feature columns: ['season', 'week', 'age', 'years_exp', 'passing_yards_r_avg_1_lag_1', 'passing_yards_r_avg_3_lag_1', 'passing_yards_r_avg_5_lag_1', 'passing_yards_r_avg_8_lag_1', 'passing_tds_r_avg_1_lag_1', 'passing_tds_r_avg_3_lag_1', 'passing_tds_r_avg_5_lag_1', 'passing_tds_r_avg_8_lag_1', 'passing_interceptions_r_avg_1_lag_1', 'passing_interceptions_r_avg_3_lag_1', 'passing_interceptions_r_avg_5_lag_1', 'passing_interceptions_r_avg_8_lag_1', 'passing_2pt_conversions_r_avg_1_lag_1', 'passing_2pt_conversions_r_avg_3_lag_1', 'passing_2pt_conversions_r_avg_5_lag_1', 'passing_2pt_conversions_r_avg_8_lag_1']
R^2: 0.4042, RMSE: 5.3912
Saved trained model to model.joblib
